In [1]:
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig
import os
# =========================
# 1. 配置
# =========================
MODEL_NAME = os.path.abspath("./Qwen3-4B-Instruct-2507")  # 按需替换
DATA_PATH = "./intent_train.jsonl"                     # 你的微调数据
OUTPUT_DIR = "./lora_output"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = False                           # 显存不够可开 4bit

# =========================
# 2. 解析 user:{user}{assistant} 格式
# =========================


def load_dataset_from_file(path: str) -> Dataset:
    messages_list = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            turns = data["messages"]
            if not turns:
                continue

            messages_list.append({"messages": turns})

    return Dataset.from_list(messages_list)







🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1015 17:01:06.012000 2059 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1015 17:01:06.043000 2059 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[xformers|WARNING]WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.13.0+cu130)
    Python  3.10.19 (you have 3.12.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more

Unsloth: xformers is installed but cannot load with torch 2.13.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# =========================
# 3. 加载模型
# =========================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

# =========================
# 4. 注入 LoRA
# =========================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


==((====))==  Unsloth 2026.10.5: Fast Qwen3 patching. Transformers: 5.16.1. vLLM: 0.27.1.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 22.184 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
==((====))==  Unsloth 2026.10.5: Fast Qwen3 patching. Transformers: 5.16.1. vLLM: 0.27.1.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 22.184 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth 2026.10.5 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [4]:
# =========================
# 5. 设置聊天模板
# =========================
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",  # 如果模型不是 Qwen2.5，改成对应模板，如 "llama-3.1"
)


def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}


# =========================
# 6. 加载数据
# =========================
dataset = load_dataset_from_file(DATA_PATH)
dataset = dataset.map(formatting_prompts_func, batched=True)




Map:   0%|          | 0/340 [00:00<?, ? examples/s]

In [ ]:
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=200,
        learning_rate=2e-4,
        optim="adamw_8bit",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_steps=50,
        output_dir=OUTPUT_DIR,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
    ),
)

# =========================
# 9. 只训练 assistant 部分
# =========================
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.train()

# =========================
# 9. 保存 LoRA 适配器
# =========================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# =========================
# 10. 保存合并后的完整模型
# =========================
model = FastLanguageModel.for_inference(model)
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./merged_model")
tokenizer.save_pretrained("./merged_model")

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/340 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/340 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 340 | Num Epochs = 5 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.116617
20,0.021440
30,0.007073
40,0.007903
50,0.005161
60,0.001666
70,0.002021
80,0.004199
90,0.001282
100,0.002137


Unsloth: Restored added_tokens_decoder metadata in ./lora_output/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./lora_output/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./lora_output/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./lora_output/checkpoint-200/tokenizer_config.json.
